In [1]:
import sys
sys.path.insert(0, "..") # Projektroot in Path
import os
os.chdir("..")

import yaml
from src.utils.seed import set_seed, get_device
from src.utils.data import make_food11_loaders
from src.models.vit import build_vit_from_cfg
from src.models.cnn_baseline import build_cnn_baseline
from src.utils.training import run_training
from src.utils.metrics import count_params, count_trainable_params

/Users/matsadel/Desktop/Master Informatik/MasterThesis/masterenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
    
set_seed(cfg["seed"])
device = get_device()
print("Device: ", device)

Device:  mps


In [3]:
train_loader, val_loader, test_loader, num_classes = make_food11_loaders(
    **cfg["dataset"]
    )

print(f"Classes: {num_classes} | Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

Classes: 11 | Train: 9866 | Val: 3430


In [ ]:
model = build_vit_from_cfg(num_classes, cfg["model"])

print(f"Params: {count_params(model)/1e6:.1f}M | Trainable: {count_trainable_params(model)/1e6:.1f}M")

best_val, history = run_training(model, train_loader, val_loader, device, epochs=1)

In [4]:
'''Cnn Baselines'''
resnet = build_cnn_baseline('resnet18.a1_in1k', num_classes)
effnet = build_cnn_baseline('efficientnet_b0.ra_in1k', num_classes)
mobilenet = build_cnn_baseline('mobilenetv3_large_100.ra_in1k', num_classes)

In [5]:
print(f"Params: {count_params(resnet)/1e6:.1f}M | Trainable: {count_trainable_params(resnet)/1e6:.1f}M")

best_val, history = run_training(resnet, train_loader, val_loader, device, epochs=1)

Params: 11.2M | Trainable: 11.2M
Epoch 001/1 | train_acc=0.6192 | val_acc=0.8064


In [6]:
print(f"Params: {count_params(mobilenet)/1e6:.1f}M | Trainable: {count_trainable_params(mobilenet)/1e6:.1f}M")

best_val, history = run_training(mobilenet, train_loader, val_loader, device, epochs=1)

Params: 4.2M | Trainable: 4.2M
Epoch 001/1 | train_acc=0.7556 | val_acc=0.8227


In [7]:
print(f"Params: {count_params(effnet)/1e6:.1f}M | Trainable: {count_trainable_params(effnet)/1e6:.1f}M")

best_val, history = run_training(effnet, train_loader, val_loader, device, epochs=1)

Params: 4.0M | Trainable: 4.0M
Epoch 001/1 | train_acc=0.7692 | val_acc=0.8638
